# Boolean oracle sketching

Compare a sampled phase sketch, inverse-probability weighting, and a fresh QSVT correction using only distribution bounds.

## Express a sampled oracle as coherent conditional phases

The editable three-bit preview below composes equality-controlled constant
phases over a uniform register. Classical sampling determines the phase list;
FlagQuantum builds each coherent predicate, its conditional global phase and
clean work. The full original 200-sample demonstration, twelve-size benchmark,
known-distribution test and fresh QSVT correction still run independently.

The correction path in [quantum.py](quantum.py) uses classically assembled
response matrices and Circuit execution. It is not a general qfunc QSVT API,
and its represented operations do not measure hardware-native gate complexity.
Try editing one preview angle and compare all complex amplitudes with the
analytic diagonal oracle. Probabilities alone remain uniform and would miss
the change. Sampling-distribution assumptions and threshold fitting are
separate from correctness of quantum predicate lowering.


Use the local FlagQuantum environment.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

import flagquantum as fq


Generate the original seeded Boolean table and the full benchmark sequence.

In [ ]:
from examples.synthesis.algorithms.quantum_primitives.quantum_oracle_sketching_boolean.overview import (
    workflow,
)
from examples.synthesis.algorithms.quantum_primitives.quantum_oracle_sketching_boolean.overview.main import (
    print_report,
    save_figures,
)

print(fq.__name__)
result = workflow.run()
print(result.benchmark.table)


Accumulate commuting phases by basis word and apply them in a native qfunc.

In [ ]:
import torch
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QNum,
    allocate,
    control,
    hadamard_transform,
    phase,
    qfunc,
)


@qfunc
def apply_phase(angle: float, word: int, data: QNum) -> None:
    control(data == word, lambda: phase(1, angle))


@qfunc
def diagonal(angles: list[float], data: QNum) -> None:
    for i in range(len(angles)):
        apply_phase(angles[i], i, data)


@qfunc
def uniform_sketch(width: int, angles: list[float], data: Output[QNum]) -> None:
    allocate(width, data)
    hadamard_transform(data)
    diagonal(angles, data)


preview_angles = [0.13 * index - 0.2 for index in range(8)]
preview = synth.synthesize(
    uniform_sketch,
    classical_inputs={"width": 3, "angles": preview_angles},
    max_qubits=5,
    max_workspace=2,
    memory_limit_bytes=4 * 1024**2,
)
preview_state = fq.run(preview).statevector().reshape(8, -1)
expected_state = (
    torch.exp(1j * torch.tensor(preview_angles, dtype=torch.float64)) / 8**0.5
)
torch.testing.assert_close(preview_state[:, 0], expected_state, atol=1e-12, rtol=0)
assert float(preview_state[:, 1:].abs().square().sum()) < 1e-24
print("Editable phase-oracle resources:", preview.report["resources"])
print("Uniform sketch resources:", result.demo.report["resources"])


Check the original 200-sample demonstration on the three-bit space.

In [ ]:
print("Demo full-state max error:", result.demo_error)


Measure the uncorrected Bell sketch with 10,000 sampled outcomes.

In [ ]:
print("Uncorrected sampled Bell fidelity:", result.bell_sampled)


Read the single-realization and mean-operator errors over all twelve sample counts.

In [ ]:
print(result.benchmark.sizes)
print(result.benchmark.mean_error)
print(result.benchmark.mean_operator_error)


Verify inverse-probability weighting against the individual phase factors.

In [ ]:
print("Known-distribution full-state max error:", result.known_error)


Fit the degree-18 threshold and check the corrected operator on all basis words.

In [ ]:
print("Threshold fit error:", result.threshold_error)
print("Corrected basis checks:", result.corrected.oracle_error)


Compare the postselected Bell fidelity and inspect the measured scaling and threshold curves.

In [ ]:
print_report(result)
figures = Path("notebook-figures")
save_figures(result, figures)
display(Image(filename=str(figures / "oracle-scaling.png")))
display(Image(filename=str(figures / "threshold-response.png")))
